In [ ]:
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from datetime import datetime

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

UPDATED WITH SOLAR AND WIND OUTPUTS!

NOTE: One modeling caution

This works well if you truly want:

one common day classification based on weather
then separate PV and wind outputs from that same classification

But if PV and wind respond to very different subsets of the weather variables, sometimes the best clustering for PV is not the best clustering for wind.

So this shared-cluster approach is great if your goal is:

one unified weather-day typology
two output profiles from it

That sounds like what you want.

In [ ]:
# 0.5 AGGREGATE POWER COLUMNS
# =========================================================

def aggregate_power_columns(
    df_input,
    pv_cols=None,
    wind_power_cols=None,
    drop_original=False
):
    """
    Aggregate multiple PV and wind power columns into total signals.
    """
    df_aggregated = df_input.copy()

    if pv_cols is not None:
        missing_pv = [col for col in pv_cols if col not in df_aggregated.columns]
        if missing_pv:
            raise ValueError(f"Missing PV columns: {missing_pv}")

        df_aggregated["PV_total"] = df_aggregated[pv_cols].sum(axis=1, min_count=1)

    if wind_power_cols is not None:
        missing_wind = [col for col in wind_power_cols if col not in df_aggregated.columns]
        if missing_wind:
            raise ValueError(f"Missing wind columns: {missing_wind}")

        df_aggregated["Wind_total"] = df_aggregated[wind_power_cols].sum(axis=1, min_count=1)

    if drop_original:
        cols_to_drop = []
        if pv_cols is not None:
            cols_to_drop.extend(pv_cols)
        if wind_power_cols is not None:
            cols_to_drop.extend(wind_power_cols)

        df_aggregated = df_aggregated.drop(columns=cols_to_drop)

    return df_aggregated

In [ ]:
# Read downloaded csv data and make df –– Last downloaded the power data in February 2026; I haven't used Jakob's new df

final_file = "data/complete_dataframe_30min.csv"
df = pd.read_csv(str(ROOT/final_file), delimiter=',', decimal='.', parse_dates=['ts'], index_col='ts') #ts is already the index :)

# Power columns to aggregate
pv_cols = ['B117_m','B319_m','B330_1_m','B330_2_m','B330_3_m','B716_m','B715_m']
wind_power_cols = ['Aircon_WT Power_m', 'Gaia_WT Power_m']

df = aggregate_power_columns(df, pv_cols, wind_power_cols,drop_original=False)

In [ ]:
df.columns

In [ ]:
# =========================================================
# 1. PREPARE DAILY PROFILES ONCE
# =========================================================

def prepare_daily_profiles_multi_output(df_input,feature_cols,target_cols,samples_per_day=48):
    """
    Validate and reshape half-hourly data into daily profiles.

    Returns
    -------
    X_daily : DataFrame
        One row per valid day, containing all daily weather values.
    y_daily_dict : dict[str, DataFrame]
        Dictionary of daily target matrices, one per target column.
    valid_days : np.ndarray
        Sorted array of valid day labels.
    """
    required_cols = feature_cols + target_cols

    df_work = df_input.copy().sort_index()
    df_work["date"] = df_work.index.date
    df_work["time"] = df_work.index.strftime("%H:%M")

    # Keep only full days
    rows_per_day = df_work.groupby("date").size()
    full_days = rows_per_day[rows_per_day == samples_per_day].index
    df_work = df_work[df_work["date"].isin(full_days)].copy()

    # Keep only days with complete data in required columns
    valid_day_mask = (
        df_work.groupby("date")[required_cols]
        .apply(lambda day_data: not day_data.isna().any().any())
    )
    valid_days = np.array(sorted(valid_day_mask[valid_day_mask].index))
    df_work = df_work[df_work["date"].isin(valid_days)].copy()

    # Build daily weather feature matrix
    daily_feature_blocks = []

    for feature_name in feature_cols:
        daily_feature_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=feature_name
        )
        daily_feature_matrix = daily_feature_matrix.reindex(
            sorted(daily_feature_matrix.columns),
            axis=1
        )
        daily_feature_matrix.columns = [
            f"{feature_name}_{time_label}"
            for time_label in daily_feature_matrix.columns
        ]
        daily_feature_blocks.append(daily_feature_matrix)

    X_daily = pd.concat(daily_feature_blocks, axis=1)

    # Build one daily target matrix per target column
    y_daily_dict = {}

    for target_col in target_cols:
        daily_target_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=target_col
        )
        daily_target_matrix = daily_target_matrix.reindex(
            sorted(daily_target_matrix.columns),
            axis=1
        )
        daily_target_matrix.columns = [
            f"{target_col}_{time_label}"
            for time_label in daily_target_matrix.columns
        ]
        y_daily_dict[target_col] = daily_target_matrix

    # Align everything on common days
    common_days = X_daily.index.copy()
    for target_col in target_cols:
        common_days = common_days.intersection(y_daily_dict[target_col].index)

    X_daily = X_daily.loc[common_days]
    for target_col in target_cols:
        y_daily_dict[target_col] = y_daily_dict[target_col].loc[common_days]

    valid_days = np.array(sorted(common_days))

    return X_daily, y_daily_dict, valid_days

## $\text{\textcolor{green}{Creation of Daily Profiles––Weather Input and Power Outputs (PV and Wind)}}$

In [ ]:
# Weather features used for clustering
# They are: 'wind_dir','wind_max', 'wind_min', 'wind_speed', 'radia_glob', 'temp_dry',
featured_weather_cols = ["temp_dry", "radia_glob", "wind_speed"]

# Two targets predicted from the same weather clusters
target_power_cols = ["PV_total", "Wind_total"]

# Candidate K values
k_values = np.arange(2, 12)

# Step 1: prepare daily profiles once
X_daily, y_daily_dict, valid_days = prepare_daily_profiles_multi_output(df, featured_weather_cols, target_power_cols, samples_per_day=48)

print(k_values)
print("Number of valid days:", len(valid_days))
print("X_daily shape:", X_daily.shape)
print("PV daily shape:", y_daily_dict["PV_total"].shape)
print("Wind daily shape:", y_daily_dict["Wind_total"].shape)
#print("X_daily Head", X_daily.head)
#print("PV_daily Head", y_daily_dict["PV_total"].head)
#print("Wind_daily Head", y_daily_dict["Wind_total"].head)


## $\text{\textcolor{green}{Choosing an Optimal K for KMeans}}$

In [ ]:
# =========================================================
# 2. CREATE MONTHLY SAMPLED-DAY FOLDS FROM valid_days
# =========================================================

def make_monthly_sampled_folds_from_days(valid_days,n_folds=5,test_days_per_month=3,random_state=42):
    """
    Create folds from an existing set of valid days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    folds = []

    for fold_id in range(n_folds):
        sampled_test_days = []

        for _, month_group in valid_day_table.groupby("month"):
            month_days = month_group["date"].dt.date.values
            n_test_days = min(test_days_per_month, len(month_days))

            chosen_test_days = rng.choice(
                month_days,
                size=n_test_days,
                replace=False
            )
            sampled_test_days.extend(chosen_test_days.tolist())

        test_days = np.array(sorted(set(sampled_test_days)))
        train_days = np.array(sorted(set(valid_days) - set(test_days)))

        folds.append({
            "fold_id": fold_id,
            "train_days": train_days,
            "test_days": test_days
        })

    return folds

In [ ]:
# =========================================================
# 3. METRIC HELPER
# =========================================================

def compute_profile_metrics(y_true_daily, y_pred_daily):
    """
    Compute global and per-day profile metrics for one target.
    """
    y_true_flat = y_true_daily.to_numpy().flatten()
    y_pred_flat = y_pred_daily.to_numpy().flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    daily_mae = np.mean(np.abs(y_true_daily.to_numpy() - y_pred_daily.to_numpy()).mean(axis=1))

    return {
        "mae": mae,
        "rmse": rmse,
        "daily_mae": daily_mae
    }

In [ ]:
# =========================================================
# 4. EVALUATE ONE K WITH SAME WEATHER CLUSTERS, TWO OUTPUTS
# =========================================================

def evaluate_kmeans_for_k_multi_output(
    X_train_daily,
    y_train_dict,
    X_test_daily,
    y_test_dict,
    k,
    random_state=42
):
    """
    Train one KMeans model on weather-day profiles and predict multiple
    daily power targets from the same cluster assignments.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_daily)
    X_test_scaled = scaler.transform(X_test_daily)

    kmeans_model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    train_cluster_labels = kmeans_model.fit_predict(X_train_scaled)
    test_cluster_labels = kmeans_model.predict(X_test_scaled)

    cluster_size_series = pd.Series(train_cluster_labels).value_counts().sort_index()

    cluster_size_info = {
        "cluster_sizes": cluster_size_series,
        "min_cluster_size": int(cluster_size_series.min()),
        "max_cluster_size": int(cluster_size_series.max()),
        "mean_cluster_size": float(cluster_size_series.mean()),
        "median_cluster_size": float(cluster_size_series.median()),
        "n_small_clusters_lt_5": int((cluster_size_series < 5).sum()),
        "n_small_clusters_lt_10": int((cluster_size_series < 10).sum())
    }
    
    cluster_profile_dict = {}
    y_pred_dict = {}
    metrics_dict = {}

    for target_name, y_train_daily in y_train_dict.items():
        y_test_daily = y_test_dict[target_name]

        train_targets_with_clusters = y_train_daily.copy()
        train_targets_with_clusters["cluster"] = train_cluster_labels

        cluster_profiles = train_targets_with_clusters.groupby("cluster").mean()
        cluster_profile_dict[target_name] = cluster_profiles

        y_pred_daily = pd.DataFrame(
            index=y_test_daily.index,
            columns=y_test_daily.columns,
            dtype=float
        )

        for day, cluster_label in zip(y_test_daily.index, test_cluster_labels):
            y_pred_daily.loc[day] = cluster_profiles.loc[cluster_label].values

        y_pred_dict[target_name] = y_pred_daily
        metrics_dict[target_name] = compute_profile_metrics(y_test_daily, y_pred_daily)

    return {
        "k": k,
        "scaler": scaler,
        "kmeans_model": kmeans_model,
        "train_cluster_labels": train_cluster_labels,
        "test_cluster_labels": test_cluster_labels,
        "cluster_profile_dict": cluster_profile_dict,
        "y_pred_dict": y_pred_dict,
        "metrics_dict": metrics_dict,
        "cluster_size_info": cluster_size_info
    }

In [ ]:
# =========================================================
# 5. EVALUATE ONE K ACROSS FIXED FOLDS
# =========================================================

def evaluate_k_across_folds_multi_output(
    X_daily,
    y_daily_dict,
    folds,
    k,
    random_state=42
):
    """
    Evaluate one candidate K across fixed folds for multiple output targets.
    """
    fold_metrics = []

    target_names = list(y_daily_dict.keys())

    for fold_definition in folds:
        train_days = fold_definition["train_days"]
        test_days = fold_definition["test_days"]

        X_train_daily = X_daily.loc[train_days]
        X_test_daily = X_daily.loc[test_days]

        y_train_dict = {
            target_name: y_daily_dict[target_name].loc[train_days]
            for target_name in target_names
        }
        y_test_dict = {
            target_name: y_daily_dict[target_name].loc[test_days]
            for target_name in target_names
        }

        fold_result = evaluate_kmeans_for_k_multi_output(
            X_train_daily=X_train_daily,
            y_train_dict=y_train_dict,
            X_test_daily=X_test_daily,
            y_test_dict=y_test_dict,
            k=k,
            random_state=random_state
        )
        
        cluster_size_info = fold_result["cluster_size_info"]
        
        row = {
            "k": k,
            "fold_id": fold_definition["fold_id"],
            "n_train_days": len(train_days),
            "n_test_days": len(test_days),
            "min_cluster_size": cluster_size_info["min_cluster_size"],
            "max_cluster_size": cluster_size_info["max_cluster_size"],
            "mean_cluster_size": cluster_size_info["mean_cluster_size"],
            "median_cluster_size": cluster_size_info["median_cluster_size"],
            "n_small_clusters_lt_5": cluster_size_info["n_small_clusters_lt_5"],
            "n_small_clusters_lt_10": cluster_size_info["n_small_clusters_lt_10"]
        }

        for target_name in target_names:
            target_metrics = fold_result["metrics_dict"][target_name]
            row[f"{target_name}_mae"] = target_metrics["mae"]
            row[f"{target_name}_rmse"] = target_metrics["rmse"]
            row[f"{target_name}_daily_mae"] = target_metrics["daily_mae"]

        fold_metrics.append(row)

    fold_results_df = pd.DataFrame(fold_metrics)

    summary = {"k": k}

    for target_name in target_names:
        summary[f"mean_{target_name}_mae"] = fold_results_df[f"{target_name}_mae"].mean()
        summary[f"std_{target_name}_mae"] = fold_results_df[f"{target_name}_mae"].std(ddof=1)
        summary[f"mean_{target_name}_rmse"] = fold_results_df[f"{target_name}_rmse"].mean()
        summary[f"std_{target_name}_rmse"] = fold_results_df[f"{target_name}_rmse"].std(ddof=1)
        summary[f"mean_{target_name}_daily_mae"] = fold_results_df[f"{target_name}_daily_mae"].mean()
        summary[f"std_{target_name}_daily_mae"] = fold_results_df[f"{target_name}_daily_mae"].std(ddof=1)

    # Combined score for choosing K: average of target daily MAEs
    summary["mean_combined_daily_mae"] = np.mean(
        [summary[f"mean_{target_name}_daily_mae"] for target_name in target_names]
    )
    
    # Cluster size summary across folds
    summary["mean_min_cluster_size"] = fold_results_df["min_cluster_size"].mean()
    summary["worst_min_cluster_size"] = fold_results_df["min_cluster_size"].min()
    summary["mean_max_cluster_size"] = fold_results_df["max_cluster_size"].mean()
    summary["mean_mean_cluster_size"] = fold_results_df["mean_cluster_size"].mean()
    summary["mean_median_cluster_size"] = fold_results_df["median_cluster_size"].mean()
    summary["mean_n_small_clusters_lt_5"] = fold_results_df["n_small_clusters_lt_5"].mean()
    summary["mean_n_small_clusters_lt_10"] = fold_results_df["n_small_clusters_lt_10"].mean()

    return fold_results_df, summary

In [ ]:
# =========================================================
# 6. SEARCH BEST K USING SAME WEATHER CLUSTERS FOR BOTH TARGETS
# =========================================================

def search_best_k_with_folds_multi_output(
    X_daily,
    y_daily_dict,
    valid_days,
    k_values,
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42
):
    """
    Generate folds once from valid_days, reuse them across all K values,
    and choose best K from combined average fold performance.
    """
    folds = make_monthly_sampled_folds_from_days(
        valid_days=valid_days,
        n_folds=n_folds,
        test_days_per_month=test_days_per_month,
        random_state=fold_random_state
    )

    all_fold_results = []
    k_summary_rows = []

    for k in k_values:
        fold_results_df, k_summary = evaluate_k_across_folds_multi_output(
            X_daily=X_daily,
            y_daily_dict=y_daily_dict,
            folds=folds,
            k=k,
            random_state=model_random_state
        )

        all_fold_results.append(fold_results_df)
        k_summary_rows.append(k_summary)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    summary_df = pd.DataFrame(k_summary_rows).sort_values("k").reset_index(drop=True)

    best_k_row = summary_df.loc[summary_df["mean_combined_daily_mae"].idxmin()]
    best_k = int(best_k_row["k"])

    return {
        "folds": folds,
        "all_fold_results": all_fold_results_df,
        "summary": summary_df,
        "best_k": best_k
    }

### $\text{\textcolor{green}{Calling on Functions}}$

In [ ]:
# Step 2: choose best K using folds
search_result = search_best_k_with_folds_multi_output(X_daily,y_daily_dict,valid_days,k_values,n_folds=5,test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42
)

best_k = search_result["best_k"]
folds = search_result["folds"]

print("Best K:", best_k)
print(search_result["summary"])

### I NEED A VISUAL REPRESENTATION OF HOW THE ERROR EVOLVES FOR PV AND WIND AS K INCREASES
### WHAT IS THE SMALLEST CLUSTER SIZE -- DOES IT IMPACT ANYTHING?

# Optional: inspect fold membership
#fold_df = folds_to_dataframe(folds)
#print(fold_df.head())

### $\text{\textcolor{green}{Plots}}$

In [ ]:
def plot_mae_vs_k(summary_df):
    """
    Plot fold-averaged daily MAE versus K for PV, wind, and combined score.
    """
    plt.figure(figsize=(8, 5))

    plt.plot(summary_df["k"], summary_df["mean_PV_total_daily_mae"], marker="o", label="PV daily MAE")
    plt.plot(summary_df["k"], summary_df["mean_Wind_total_daily_mae"], marker="o", label="Wind daily MAE")
    plt.plot(summary_df["k"], summary_df["mean_combined_daily_mae"], marker="o", label="Combined daily MAE")

    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Mean daily MAE across folds")
    plt.title("MAE vs K")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_min_cluster_size_vs_k(summary_df):
    """
    Plot how the minimum cluster size evolves with K.
    """
    plt.figure(figsize=(8, 5))

    plt.plot(
        summary_df["k"],
        summary_df["worst_min_cluster_size"],
        marker="o",
        label="Lowest min cluster size"
    )

    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Minimum cluster size")
    plt.title("Minimum cluster size vs K")
    plt.legend()
    plt.grid(True)

    plt.show()

# CALLING ON THE PLOT FUNCTION ABOVE DEFINED :=)
plot_mae_vs_k(search_result["summary"])
plot_min_cluster_size_vs_k(search_result["summary"])


## $\text{\textcolor{green}{Final Model with Optimal K}}$

### $\text{\textcolor{green}{Functions}}$

In [ ]:
# 7. CREATE A FRESH FINAL MONTHLY-BALANCED SPLIT
# =========================================================

def create_final_monthly_split_from_days(
    valid_days,
    test_days_per_month=3,
    random_state=999
):
    """
    Create one fresh final split from valid_days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    sampled_final_test_days = []

    for _, month_group in valid_day_table.groupby("month"):
        month_days = month_group["date"].dt.date.values
        n_test_days = min(test_days_per_month, len(month_days))

        chosen_test_days = rng.choice(
            month_days,
            size=n_test_days,
            replace=False
        )
        sampled_final_test_days.extend(chosen_test_days.tolist())

    final_test_days = np.array(sorted(set(sampled_final_test_days)))
    final_train_days = np.array(sorted(set(valid_days) - set(final_test_days)))

    return {
        "train_days": final_train_days,
        "test_days": final_test_days
    }

In [ ]:
# 8. FINAL MODEL FIT
# =========================================================

def fit_final_model_multi_output(
    X_daily,
    y_daily_dict,
    valid_days,
    best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
):
    """
    After best K is selected:
    - create a fresh final split
    - retrain/evaluate with chosen K
    """
    final_split = create_final_monthly_split_from_days(
        valid_days=valid_days,
        test_days_per_month=final_test_days_per_month,
        random_state=final_split_random_state
    )

    final_train_days = final_split["train_days"]
    final_test_days = final_split["test_days"]

    X_train_daily = X_daily.loc[final_train_days]
    X_test_daily = X_daily.loc[final_test_days]

    y_train_dict = {
        target_name: target_matrix.loc[final_train_days]
        for target_name, target_matrix in y_daily_dict.items()
    }
    y_test_dict = {
        target_name: target_matrix.loc[final_test_days]
        for target_name, target_matrix in y_daily_dict.items()
    }

    final_model_result = evaluate_kmeans_for_k_multi_output(
        X_train_daily=X_train_daily,
        y_train_dict=y_train_dict,
        X_test_daily=X_test_daily,
        y_test_dict=y_test_dict,
        k=best_k,
        random_state=model_random_state
    )

    return {
        "best_k": best_k,
        "final_train_days": final_train_days,
        "final_test_days": final_test_days,
        "X_train_daily": X_train_daily,
        "X_test_daily": X_test_daily,
        "y_train_dict": y_train_dict,
        "y_test_dict": y_test_dict,
        "model_result": final_model_result
    }

### $\text{\textcolor{green}{Calling on the functions}}$

In [ ]:
# Step 3: fit final model with fresh final split
final_result = fit_final_model_multi_output(
    X_daily=X_daily,
    y_daily_dict=y_daily_dict,
    valid_days=valid_days,
    best_k=best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
)

print("Final PV daily MAE:", final_result["model_result"]["metrics_dict"]["PV_total"]["daily_mae"])
print("Final Wind daily MAE:", final_result["model_result"]["metrics_dict"]["Wind_total"]["daily_mae"])
print("Final PV RMSE:", final_result["model_result"]["metrics_dict"]["PV_total"]["rmse"])
print("Final Wind RMSE:", final_result["model_result"]["metrics_dict"]["Wind_total"]["rmse"])

### $\text{\textcolor{green}{Plots}}$

In [ ]:
# =========================================================
# PLOTS FOR BEST K
# =========================================================

# 1. CLUSTER SPREAD

def plot_final_cluster_spread(
    final_result,
    radiation_feature="radia_glob",
    wind_feature="windspeed",
    include_test_days=True
):
    """
    Plot cluster spread for the final fitted model using daily mean
    radiation and daily mean windspeed.

    Parameters
    ----------
    final_result : dict
        Output from fit_final_model_multi_output(...)
    radiation_feature : str
        Base name of radiation feature in X_daily, e.g. "radia_glob"
    wind_feature : str
        Base name of wind feature in X_daily, e.g. "windspeed"
    include_test_days : bool
        If True, include both train and test days in the scatter plot.
    """
    X_train_daily = final_result["X_train_daily"]
    X_test_daily = final_result["X_test_daily"]

    model_result = final_result["model_result"]
    train_cluster_labels = pd.Series(
        model_result["train_cluster_labels"],
        index=X_train_daily.index,
        name="cluster"
    )
    test_cluster_labels = pd.Series(
        model_result["test_cluster_labels"],
        index=X_test_daily.index,
        name="cluster"
    )

    radiation_cols_train = [col for col in X_train_daily.columns if col.startswith(f"{radiation_feature}_")]
    wind_cols_train = [col for col in X_train_daily.columns if col.startswith(f"{wind_feature}_")]

    if len(radiation_cols_train) == 0:
        raise ValueError(f"No columns found for radiation feature '{radiation_feature}'")
    if len(wind_cols_train) == 0:
        raise ValueError(f"No columns found for wind feature '{wind_feature}'")

    train_plot_df = pd.DataFrame({
        "daily_mean_radiation": X_train_daily[radiation_cols_train].mean(axis=1),
        "daily_mean_windspeed": X_train_daily[wind_cols_train].mean(axis=1),
        "cluster": train_cluster_labels,
        "set": "train"
    }, index=X_train_daily.index)

    plot_frames = [train_plot_df]

    if include_test_days:
        radiation_cols_test = [col for col in X_test_daily.columns if col.startswith(f"{radiation_feature}_")]
        wind_cols_test = [col for col in X_test_daily.columns if col.startswith(f"{wind_feature}_")]

        test_plot_df = pd.DataFrame({
            "daily_mean_radiation": X_test_daily[radiation_cols_test].mean(axis=1),
            "daily_mean_windspeed": X_test_daily[wind_cols_test].mean(axis=1),
            "cluster": test_cluster_labels,
            "set": "test"
        }, index=X_test_daily.index)

        plot_frames.append(test_plot_df)

    plot_df = pd.concat(plot_frames)

    plt.figure(figsize=(8, 6))

    for cluster_id in sorted(plot_df["cluster"].unique()):
        cluster_data = plot_df[plot_df["cluster"] == cluster_id]
        plt.scatter(
            cluster_data["daily_mean_radiation"],
            cluster_data["daily_mean_windspeed"],
            label=f"Cluster {cluster_id}",
            alpha=0.7
        )

    plt.xlabel(f"Daily mean {radiation_feature}")
    plt.ylabel(f"Daily mean {wind_feature}")
    plt.title("Final model cluster spread")
    plt.legend()
    plt.grid(True)
    plt.show()

# 2. AVERAGE DAY PROFILE PER CLUSTER 

def plot_final_cluster_average_profiles(final_result, target_name):
    """
    Plot the average daily profile for each cluster for one target.

    target_name should be one of:
    - "PV_total"
    - "Wind_total"
    """
    cluster_profiles = final_result["model_result"]["cluster_profile_dict"][target_name]

    plt.figure(figsize=(10, 5))

    for cluster_id in cluster_profiles.index:
        plt.plot(
            range(cluster_profiles.shape[1]),
            cluster_profiles.loc[cluster_id].values,
            label=f"Cluster {cluster_id}"
        )

    plt.xlabel("Half-hour step of day")
    plt.ylabel(target_name)
    plt.title(f"Average daily {target_name} profile by cluster")
    plt.legend()
    plt.grid(True)
    plt.show()

# 3. PREDICTED VS TRUES CURVES FOR PV AND WIND

def plot_final_true_vs_predicted_day(final_result, day=None):
    """
    Plot predicted vs true PV and wind curves for one day in the final test set.

    If day is None, the first final test day is used.
    """
    y_test_dict = final_result["y_test_dict"]
    y_pred_dict = final_result["model_result"]["y_pred_dict"]

    available_days = y_test_dict["PV_total"].index

    if day is None:
        day = available_days[0]

    if day not in available_days:
        raise ValueError(f"Day {day} is not in the final test set.")

    pv_true = y_test_dict["PV_total"].loc[day].values
    pv_pred = y_pred_dict["PV_total"].loc[day].values

    wind_true = y_test_dict["Wind_total"].loc[day].values
    wind_pred = y_pred_dict["Wind_total"].loc[day].values

    plt.figure(figsize=(10, 5))
    plt.plot(range(len(pv_true)), pv_true, label="PV true")
    plt.plot(range(len(pv_pred)), pv_pred, label="PV predicted")
    plt.xlabel("Half-hour step of day")
    plt.ylabel("PV_total")
    plt.title(f"PV: true vs predicted for {day}")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(range(len(wind_true)), wind_true, label="Wind true")
    plt.plot(range(len(wind_pred)), wind_pred, label="Wind predicted")
    plt.xlabel("Half-hour step of day")
    plt.ylabel("Wind_total")
    plt.title(f"Wind: true vs predicted for {day}")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_final_cluster_spread(
    final_result,
    radiation_feature="radia_glob",
    wind_feature="wind_speed",
    include_test_days=True
)

plot_final_cluster_average_profiles(final_result, "PV_total")
plot_final_cluster_average_profiles(final_result, "Wind_total")

plot_final_true_vs_predicted_day(final_result)


In [ ]:
# =========================================================
# OPTIONAL: EXPORT / INSPECT FOLDS
# =========================================================

def folds_to_dataframe(folds):
    fold_rows = []

    for fold_definition in folds:
        fold_id = fold_definition["fold_id"]

        for train_day in fold_definition["train_days"]:
            fold_rows.append({
                "date": pd.to_datetime(train_day),
                "fold_id": fold_id,
                "role": "train"
            })

        for test_day in fold_definition["test_days"]:
            fold_rows.append({
                "date": pd.to_datetime(test_day),
                "fold_id": fold_id,
                "role": "test"
            })

    fold_df = pd.DataFrame(fold_rows).sort_values(["fold_id", "date"]).reset_index(drop=True)
    return fold_df


def save_folds_to_csv(folds, filepath):
    fold_df = folds_to_dataframe(folds)
    fold_df.to_csv(filepath, index=False)


def load_folds_from_csv(filepath):
    fold_df = pd.read_csv(filepath, parse_dates=["date"])

    folds = []

    for fold_id, fold_group in fold_df.groupby("fold_id"):
        train_days = fold_group.loc[
            fold_group["role"] == "train", "date"
        ].dt.date.values

        test_days = fold_group.loc[
            fold_group["role"] == "test", "date"
        ].dt.date.values

        folds.append({
            "fold_id": int(fold_id),
            "train_days": np.array(sorted(train_days)),
            "test_days": np.array(sorted(test_days))
        })

    return folds